In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = ""

In [2]:
!pip install -q langchain-core requests


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install -q langchain_google_genai


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

/workspaces/langchain-implementations/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# tool create

@tool
def multiply(a : int, b:int) -> int:
    """Given two numbers a and b this tool returns their product """
    return a*b

In [7]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [8]:
multiply.name

'multiply'

In [9]:
multiply.description

'Given two numbers a and b this tool returns their product'

In [10]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [19]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [22]:
llm_with_tools = llm.bind_tools([multiply])

In [23]:
llm_with_tools.invoke('hi')

AIMessage(content='Hello! I can help you multiply two numbers. What numbers would you like to multiply?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec04a-4d3f-7313-b11c-82e6ece50ac7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 18, 'total_tokens': 72, 'input_token_details': {'cache_read': 0}})

In [24]:
llm_with_tools.invoke('can you multiply 12 with 15')

AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 15, "a": 12}'}, '__gemini_function_call_thought_signatures__': {'e3267884-b866-4dc8-b08b-b5cf040832ac': 'CsYBAQw51sdnA0sKZcfpOQmNeJ7BcNmv+K7gCvuf4a3Q6rtGNXml2ZjKS1HcLfnGr0NGCAfVjH3k3UwgAXvymUjarArNG3cqdUsNawLduizVs8xjL3xaZsnhK1a6llWQ1GnGV2n+NRabWMOTvFvEx+9ekU/y00kzyKfMfYRr2FoQH0lEdTw1/K+vuYeVBd16gKQxL8lfz4Qe8tjNcHJgCdVzi11ixfnqAd8lw0t4BgsrhILVaY7Hk1DwiVA6IbNH1YRq082nLsQT'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec04c-695d-7693-81f2-0c7704d6bd49-0', tool_calls=[{'name': 'multiply', 'args': {'b': 15, 'a': 12}, 'id': 'e3267884-b866-4dc8-b08b-b5cf040832ac', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 71, 'total_tokens': 134, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 51}})

In [25]:
llm_with_tools.invoke('can you multiply 12 with 15').tool_calls

[{'name': 'multiply',
  'args': {'a': 12, 'b': 15},
  'id': '108b15af-97cd-485b-93ec-7960e239b6e1',
  'type': 'tool_call'}]

In [26]:
llm_with_tools.invoke('can you multiply 12 with 15').tool_calls[0]

{'name': 'multiply',
 'args': {'a': 12, 'b': 15},
 'id': '21ba857e-28ca-41a3-af10-2f06ee3c9d55',
 'type': 'tool_call'}

In [27]:
llm_with_tools.invoke('can you multiply 12 with 15').tool_calls[0]['args']

{'a': 12, 'b': 15}

In [29]:
multiply.invoke(llm_with_tools.invoke('can you multiply 12 with 15').tool_calls[0]['args'])

180

In [28]:
multiply.invoke(llm_with_tools.invoke('can you multiply 12 with 15').tool_calls[0])

ToolMessage(content='180', name='multiply', tool_call_id='91127d2f-06a8-4a00-9c9e-5609de0908fa')

In [41]:
query = HumanMessage('can you multiply 12 with 15')

In [42]:
messages = [query]

In [43]:
messages

[HumanMessage(content='can you multiply 12 with 15', additional_kwargs={}, response_metadata={})]

In [44]:
result = llm_with_tools.invoke(messages)

In [45]:
messages.append(result)

In [46]:
messages

[HumanMessage(content='can you multiply 12 with 15', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 12, "b": 15}'}, '__gemini_function_call_thought_signatures__': {'1163e6c1-b470-4841-aada-57f99ec9843e': 'CvcBAQw51sfyIxILueQUPWoC5MsxxNkbI+x1WvT8DbpVE4J8Euu+a2/M8ULG7euzHf2tQXJ+lhahePhllkoMMOUUkPlYC154bLf5jMgbgTi7ZDMdbmEKVgmrf1ybQiN5JSKWDu3tihMsWAIoXxIxAm3OS5YbHHjS4F4fXLvLKqX+m96U8czmNNsCkwdATYM/s34JvkKoCJK0wEfrY0rUSSeqXXIuleLFz9dtDI50c5uqFwGw79115XYd6WLsG0OJ0RgnmDWx3egvnAwJARcRNnqiYa7k1LSKM0LNHB+wpciUFkxBTqOj8TU9mY2qklgIKC/+wyN11SJapw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec05b-9a3c-7791-8973-5a652f7da870-0', tool_calls=[{'name': 'multiply', 'args': {'a': 12, 'b': 15}, 'id': '1163e6c1-b470-4841-aada-57f99ec9843e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata=

In [47]:
tool_result = multiply.invoke(result.tool_calls[0])

In [48]:
messages.append(tool_result)

In [49]:
messages

[HumanMessage(content='can you multiply 12 with 15', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 12, "b": 15}'}, '__gemini_function_call_thought_signatures__': {'1163e6c1-b470-4841-aada-57f99ec9843e': 'CvcBAQw51sfyIxILueQUPWoC5MsxxNkbI+x1WvT8DbpVE4J8Euu+a2/M8ULG7euzHf2tQXJ+lhahePhllkoMMOUUkPlYC154bLf5jMgbgTi7ZDMdbmEKVgmrf1ybQiN5JSKWDu3tihMsWAIoXxIxAm3OS5YbHHjS4F4fXLvLKqX+m96U8czmNNsCkwdATYM/s34JvkKoCJK0wEfrY0rUSSeqXXIuleLFz9dtDI50c5uqFwGw79115XYd6WLsG0OJ0RgnmDWx3egvnAwJARcRNnqiYa7k1LSKM0LNHB+wpciUFkxBTqOj8TU9mY2qklgIKC/+wyN11SJapw=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec05b-9a3c-7791-8973-5a652f7da870-0', tool_calls=[{'name': 'multiply', 'args': {'a': 12, 'b': 15}, 'id': '1163e6c1-b470-4841-aada-57f99ec9843e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata=

In [50]:
llm_with_tools.invoke(messages)

AIMessage(content='The product of 12 and 15 is 180.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec05b-c102-7510-afae-5834cfd418d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 16, 'total_tokens': 114, 'input_token_details': {'cache_read': 0}})

In [51]:
llm_with_tools.invoke(messages).content

'The product of 12 and 15 is 180.'

In [ ]:
exchange_rate_api_key = ""

In [144]:
llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash')


In [145]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/{exchange_rate_api_key}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [146]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781308801,
 'time_last_update_utc': 'Sat, 13 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1781395201,
 'time_next_update_utc': 'Sun, 14 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.3078}

In [147]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95.3078})

953.078

In [148]:
# tool binding
llm_with_toolss = llm.bind_tools([get_conversion_factor, convert])

In [160]:
messages1 = [HumanMessage("Use the tools in this exact order and return both tool calls in one response: first call get_conversion_factor with base_currency='USD' and target_currency='INR', then call convert with base_currency_value=10.")]

In [161]:
messages1

[HumanMessage(content="Use the tools in this exact order and return both tool calls in one response: first call get_conversion_factor with base_currency='USD' and target_currency='INR', then call convert with base_currency_value=10.", additional_kwargs={}, response_metadata={})]

In [162]:
ai_message = llm_with_toolss.invoke(messages1)

In [163]:
ai_message

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}, '__gemini_function_call_thought_signatures__': {'o865f7pu': 'EucCCuQCAQw51scxAeoVxy3L9Pu2MiSxlzRdQEC8BLvkJD1E8/yr36DIqyotFyJt5wE5At+F63sBysZGc5T1aV4RwOM4CyQY9ur06hoLkiyhrF3RTjUUQdlUzzmCIvbydNQDnJ/Hew+9YRvyYuNixvJIe9yp7IpBCO3bELL26LifuCJXU0/5bLd3HWR/58XSMOXWjVB6j/jfwvK4g53mSZxrS/ggpL/BP1b9MmS0SM85QB9mPRSJp9z5zfmbsiUkHERd3yRF4RCE0J9BgcSw8x/JI6XFiaQZFOGaCE5ZxGvoYPjfUGcg8a/RTfmPc3YfeYdmDtlnOf2o4zPPaAomipxJWVWelIGGNMDoYwbMEPSSgxgR+z0iFyGjXdL4EzcOP1I5mU2TA8KVzirkg6wtjFJu0z1WfuPrblJce5yqQhajU8aA8kVkmAoVQ4o4EK7tvs5wIcPPKdp/AerSQ87UjNoGmxnK8QG+g0Q='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec0d3-150c-7823-8609-8d4fc52ecd8d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'o865f7pu', 'type': 'tool_c

In [164]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'INR', 'base_currency': 'USD'},
  'id': 'o865f7pu',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'hdu1j3mu',
  'type': 'tool_call'}]

In [165]:
messages1.append(ai_message)

In [166]:
for tool_call in ai_message.tool_calls:
    print(tool_call)

{'name': 'get_conversion_factor', 'args': {'target_currency': 'INR', 'base_currency': 'USD'}, 'id': 'o865f7pu', 'type': 'tool_call'}
{'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'hdu1j3mu', 'type': 'tool_call'}


In [167]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages1.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages1.append(tool_message2)


In [168]:
messages1

[HumanMessage(content="Use the tools in this exact order and return both tool calls in one response: first call get_conversion_factor with base_currency='USD' and target_currency='INR', then call convert with base_currency_value=10.", additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}, '__gemini_function_call_thought_signatures__': {'o865f7pu': 'EucCCuQCAQw51scxAeoVxy3L9Pu2MiSxlzRdQEC8BLvkJD1E8/yr36DIqyotFyJt5wE5At+F63sBysZGc5T1aV4RwOM4CyQY9ur06hoLkiyhrF3RTjUUQdlUzzmCIvbydNQDnJ/Hew+9YRvyYuNixvJIe9yp7IpBCO3bELL26LifuCJXU0/5bLd3HWR/58XSMOXWjVB6j/jfwvK4g53mSZxrS/ggpL/BP1b9MmS0SM85QB9mPRSJp9z5zfmbsiUkHERd3yRF4RCE0J9BgcSw8x/JI6XFiaQZFOGaCE5ZxGvoYPjfUGcg8a/RTfmPc3YfeYdmDtlnOf2o4zPPaAomipxJWVWelIGGNMDoYwbMEPSSgxgR+z0iFyGjXdL4EzcOP1I5mU2TA8KVzirkg6wtjFJu0z1WfuPrblJce5yqQhajU8aA8kVkmAoVQ4o4EK7tvs5wIcPPKdp/AerSQ87UjNoGmxnK8QG+g0Q='}}, response_metadata={'finish_reason': 'STOP', 'm

In [169]:
llm_with_toolss.invoke(messages1)

AIMessage(content=[{'type': 'text', 'text': 'I have successfully executed the tool calls as requested.\n\n1. **Get Conversion Factor**: The conversion rate from **USD** to **INR** is **95.3078**.\n2. **Convert**: Converting a base currency value of **10 USD** with the retrieved rate yields **953.078 INR**.', 'extras': {'signature': 'EjQKMgEMOdbHSHleF+EVQgi40hnQMmJQnqOyQhhl58iAvoaku48bVa0LqQeNCA+iisl4/RKO'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ec0d3-2175-7322-9b7f-75931a9e76e2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 533, 'output_tokens': 71, 'total_tokens': 604, 'input_token_details': {'cache_read': 0}})

In [170]:
llm_with_toolss.invoke(messages1).content

[{'type': 'text',
  'text': 'The first tool call, `get_conversion_factor`, retrieved a conversion rate of 95.3078 from USD to INR.\n\nThe second tool call, `convert`, used that conversion rate (approximated as 95.3078) with a base value of 10 USD to calculate the equivalent value of 953.078 INR.',
  'extras': {'signature': 'EjQKMgEMOdbHs9mBm+M8q2PAswcafzsKHBtRLZHZf5iF8ywWHKTmBlDUNdh+jqeDyS/UVJml'}}]